# Neptune Analytics with Snowflake Data via Athena Federated Query

This notebook demonstrates how to connect PaySim data stored in Snowflake and, using Athena Federated Query, create a graph view of the data in Neptune Analytics. We will:
1. Upload PaySim data to Snowflake and create a table
2. Ensure we have access rights to read the Snowflake data
3. Set up AWS Secrets Manager for Snowflake credentials
4. Configure an Athena datasource with a Snowflake connector
4. Import data from Snowflake into Neptune Analytics
5. Run Louvain algorithm for fraud detection

## Setup

Import necessary libraries and configure logging.

In [ ]:
%pip install kagglehub snowflake-connector-python

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import os
import kagglehub
import boto3
from pathlib import Path

from nx_neptune.session_manager import SessionManager
from nx_neptune.instance_management import execute_athena_query
from nx_neptune.utils.utils import get_stdout_logger, validate_and_get_env

In [ ]:
logger = get_stdout_logger(__name__,[
    'nx_neptune.instance_management',
    'nx_neptune.session_manager',
    __name__])

# Ignore cache warnings
nx.config.warnings_to_ignore.add("cache")

## Configuration

Set up environment variables for Snowflake connection and AWS resources.

In [ ]:
def check_env_vars(var_names):
    values = {}
    for var_name in var_names:
        value = os.getenv(var_name)
        if not value:
            print(f"Warning: Environment Variable {var_name} is not defined")
            print(f"You can set it using: %env {var_name}=your-value")
        else:
            print(f"Using {var_name}: {value}")
        values[var_name] = value
    return values
    
env_vars = check_env_vars([
    'NETWORKX_S3_IMPORT_BUCKET_PATH',
    'NETWORKX_S3_EXPORT_BUCKET_PATH',
    'SNOWFLAKE_DATABASE',
    'SNOWFLAKE_SCHEMA',
    'ATHENA_CATALOG_NAME',
    'ATHENA_SPILL_BUCKET',
])

env_vars = validate_and_get_env(
    'NETWORKX_S3_IMPORT_BUCKET_PATH',
    'NETWORKX_S3_EXPORT_BUCKET_PATH',
    'SNOWFLAKE_DATABASE',
    'SNOWFLAKE_SCHEMA',
    'SNOWFLAKE_TABLE'
    'ATHENA_CATALOG_NAME',
    'ATHENA_SPILL_BUCKET'
)
(
    # S3 bucket variables used for import/export
    s3_location_import,
    s3_location_export,
    # Snowflake variables
    snowflake_database,
    snowflake_schema,
    snowflake_table,
    # catalog name in Amazon Athena
    athena_snowflake_catalog,
    # spill bucket
    athena_spill_bucket
 ) = env_vars.values()

# name of the session
session_name = "nx-snowflake-demo"


## Step 1: Upload PaySim Data to Snowflake

Download PaySim dataset and upload to Snowflake. You can also perform these tasks through the Snowflake UI.  If you're using the free trial Snowflake account, you can use the provided lakehouse tutorial warehouse, database, and role:
- warehouse: `SNOWFLAKE_LEARNING_WH`
- database: `ICEBERG_TUTORIAL_DB`
- role: `SNOWFLAKE_LEARNING_ROLE`

This notebook also requires that you upload the PaySim dataset to your Snowflake account.  If you're using the free trial Snowflake account, you can trim the file for the first 100K rows and upload that (otherwise the dataset is larger than the allowed upload).

Alternatively, follow [Tutorial: Create Your First Apache Iceberg™ Table](https://docs.snowflake.com/en/user-guide/tutorials/create-your-first-iceberg-table) tutorial in Snowflake to set up an Amazon S3 bucket as an external Iceberg table for your PAYSIM data.

The user/password needed for this connector requires access to create roles, users, databases, schemas and tables.  If using the free trial Snowflake account, this can be the account admin user used to create the Snowflake account. You can add this same user/password to Amazon secrets or create a new user to access the data.

You'll need the Snowflake Python connector installed:
```bash
pip install snowflake-connector-python
```

In [ ]:
import snowflake.connector

# The Snowflake account, as in the url for the account must be https://<account>.snowflakecomputing.com
# you can find your account url under the Admin tab
snowflake_account = os.getenv('SNOWFLAKE_ACCOUNT')
snowflake_warehouse = os.getenv('SNOWFLAKE_WAREHOUSE', 'SNOWFLAKE_LEARNING_WH')
snowflake_user = os.getenv('SNOWFLAKE_USER')
snowflake_password = os.getenv('SNOWFLAKE_PASSWORD')

# Connect to Snowflake using the snowflake connection
conn = snowflake.connector.connect(
    user=snowflake_user,
    password=snowflake_password,
    account=snowflake_account,
    warehouse=snowflake_warehouse,
    database=snowflake_database,
    schema=snowflake_schema
)

cursor = conn.cursor()

Create the database (unless using the `ICEBERG_TUTORIAL_DB`), schema and table.

For this tutorial, you can consider using the following other variables:
- schema: `PAYSIM`
- table: `TRANSACTIONS`

In [ ]:
# Create database and schema if they don't exist
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {snowflake_database}")
cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {snowflake_database}.{snowflake_schema}")
cursor.execute(f"USE SCHEMA {snowflake_database}.{snowflake_schema}")

# Create PaySim transactions table
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {snowflake_table} (
   STEP INT,
   TYPE VARCHAR(50),
   AMOUNT FLOAT,
   NAMEORIG VARCHAR(100),
   OLDBALANCEORG FLOAT,
   NEWBALANCEORIG FLOAT,
   NAMEDEST VARCHAR(100),
   OLDBALANCEDEST FLOAT,
   NEWBALANCEDEST FLOAT,
   ISFRAUD INT,
   ISFLAGGEDFRAUD INT
   ) \
"""
cursor.execute(create_table_sql)

In [ ]:
# Download PaySim dataset
paysim_path = Path(kagglehub.dataset_download("ealaxi/paysim1"))
print("Path to paysim dataset files:", paysim_path)

# Upload CSV file to Snowflake stage and load into table
csv_file = next(paysim_path.glob('*.csv'))
cursor.execute("CREATE OR REPLACE STAGE paysim_stage")
cursor.execute(f"PUT file://{csv_file} @paysim_stage")
cursor.execute(f"""
COPY INTO {snowflake_table}
FROM @paysim_stage
FILE_FORMAT = (TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1)
ON_ERROR = 'CONTINUE'
""")

In [ ]:
# Verify data loaded
QUERY=f"SELECT COUNT(*) FROM {snowflake_table}"
cursor.execute(QUERY)
count = cursor.fetchone()[0]
print(f"Found {count} transactions in Snowflake")

In [ ]:
# close connection
cursor.close()
conn.close()

## Step 2: Set Up AWS Secrets Manager for Snowflake Credentials

Create a secret in AWS Secrets Manager to store Snowflake connection credentials. The Secret created will contain the user credentials needed to access the database.schema.tables.  Ensure that they have access rights to the tables in the account.  If using the account admin, they will have access rights to the tables by default.  If using another user role (such as `SNOWFLAKE_LEARNING_ROLE`) ensure:
- The user has the necessary role assigned
- The database.schema.table has SELECT access rights for the role
- Ensure that a security policy does not block access for the athena connector

**Create the secret using AWS CLI:**
```bash
aws secretsmanager create-secret \
    --name snowflake-credentials \
    --secret-string '{"user":"YOUR_SNOWFLAKE_USER","password":"YOUR_SNOWFLAKE_PASSWORD"}'
```

**Or via AWS Console:**
1. Go to AWS Secrets Manager → Secrets → Store a new secret
2. Select "Other type of secret"
3. Add key-value pairs:
   - Key: `user`, Value: Your Snowflake username
   - Key: `password`, Value: Your Snowflake password
4. Name: `snowflake-credentials`
5. Create secret and copy the ARN

In [ ]:
# Verify Secrets Manager secret exists
secrets_client = boto3.client('secretsmanager')

# Amazon Secret Manager ARN used to store the above credentials
snowflake_secret_arn = os.getenv('SNOWFLAKE_SECRET_ARN')

if snowflake_secret_arn:
    try:
        response = secrets_client.describe_secret(SecretId=snowflake_secret_arn)
        print(f"Secret found: {response['Name']}")
        print(f"ARN: {response['ARN']}")
    except secrets_client.exceptions.ResourceNotFoundException:
        print(f"ERROR: Secret not found: {snowflake_secret_arn}")
        print("Please create the secret as described above")
else:
    print("WARNING: SNOWFLAKE_SECRET_ARN not set")
    print("Set it using: %env SNOWFLAKE_SECRET_ARN=arn:aws:secretsmanager:region:account:secret:name")

## Step 3: Create Athena Data Catalog for Snowflake

Create an Athena data catalog that automatically deploys a Lambda connector for Snowflake.

**Setup via AWS Console:**

1. Go to **Athena Console** → **Data sources and catalogs** → **Create data source**
2. Select **Query a data source** → **Snowflake**
3. Provide connection parameters:
   - **Data source name**: `snowflake_catalog`
   - **Description** (optional): `PaySIM Glue Connection to Snowflake account`
   - **Host**: Your connection string (e.g., `<account>.snowflakecomputing.com`)
   - **Port**: Connection port (default: `443`)
   - **Warehouse**: Your Snowflake warehouse name (e.g. `SNOWFLAKE_LEARNING_WH`)
   - **Database**: Your Snowflake database name (e.g. `ICEBERG_TUTORIAL_DB`)
   - **Schema**: Your Snowflake schema (e.g. `PAYSIM`)
   - **JDBC Paramebers**: This can be left blank
   - **AWS Secret**: Select `snowflake-credentials`
   - **Spill location**: S3 bucket for large query results
   - Optionall:  The Encryption and Network settings for the S3 bucket may be necessary for your setup
4. Click **Create data source**
   - Athena will automatically create and configure the Lambda connector for you.  Once complete, you should see the Database appear under **Associated databases**.
5. Go to **Athena Console** → **Query Editor**, and select the data source:
   - Data source: `snowflake_catalog`
   - Database: select your schema (e.g. `PAYSIM`)
   - Your tables should be discovered automatically
   - Select the `...` and **Preview Table** to see if Athena can connect to the table

Troubleshooting
1. If the Database, Schema, or Tables are not automatically discovered, this is probably an access right issue on the user defined in the Amazon Secret. Ensure that the role assigned to the user has SELECT access to the Data, Schema, and Table.  Also, ensure that there is no Security Network Policy that blocks access to the account.
2. If the Tables are discovered, but the Query return a "TABLE_NOT_FOUND" error, this is probably due to Snowflake schemas being case-sensitive.  The Athena connector defaults to using lower-case schemas, while the Athena table discoverer defaults to upper-case schemas.  You can change the case to upper by following these steps:
    - Go to **Athena Console** → **Data sources and catalogs** → Open your datasource (e.g. `snowflake_catalog`)
    - Open the **Lambda Function** (e.g. `athenafederationcatalog_snowflake_catalog`)
    - Open the **Configuration** → **Environment variables** → **Edit**
    - Add `casing_mode`: `UPPER` variable (or `LOWER` if your schema is in lower-case)

In [ ]:
# Verify Athena catalog exists
athena_client = boto3.client('athena')
import pprint
from time import sleep

TEST_SNOWFLAKE_TABLE = f"""
SELECT DISTINCT "~id", 'customer' AS "~label"
FROM (
    SELECT NAMEORIG as "~id" FROM {snowflake_table} WHERE NAMEORIG IS NOT NULL
    UNION ALL
    SELECT NAMEDEST as "~id" FROM {snowflake_table} WHERE NAMEDEST IS NOT NULL
)
"""

import traceback

try:
    response = athena_client.get_data_catalog(Name=athena_snowflake_catalog)
    print(f"Athena catalog '{athena_snowflake_catalog}' found")
    print(f"Catalog type: {response['DataCatalog']['Type']}")
    if 'Parameters' in response['DataCatalog']:
        print(f"Lambda function: {response['DataCatalog']['Parameters'].get('function', 'N/A')}")
    
    # Execute test query using passthrough query syntax for Snowflake
    # using the athena snowflake catalog, and snowflake schema as the database
    result = execute_athena_query(
        TEST_SNOWFLAKE_TABLE,
        athena_spill_bucket,
        catalog=athena_snowflake_catalog,
        database=snowflake_schema,
        client=athena_client,
    )
    print(f"\nQuery result:")
    pprint.pprint(result)
        
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {str(e)}")
    print("\nFull traceback:")
    traceback.print_exc()

## Create Neptune Analytics Instance

Provision a Neptune Analytics instance or retrieve an existing one.

In [ ]:
session = SessionManager.session(session_name)
graph_list = session.list_graphs()
print("Available graphs:")
for g in graph_list:
    print(g)

In [ ]:
graph = await session.get_or_create_graph(config={"provisionedMemory": 16})
print(f"Retrieved graph: {graph}")

## Import Data from Snowflake via Athena Federated Query

Query Snowflake data through Athena using the Lambda connector and import into Neptune Analytics.

In [ ]:
# SQL queries to project nodes and edges from Snowflake using passthrough queries
# Set warehouse, database, and schema context before querying
SOURCE_AND_DESTINATION_CUSTOMERS = f"""
SELECT DISTINCT "~id", 'customer' AS "~label"
FROM (
    SELECT NAMEORIG as "~id" FROM {snowflake_table} WHERE NAMEORIG IS NOT NULL
    UNION ALL
    SELECT NAMEDEST as "~id" FROM {snowflake_table} WHERE NAMEDEST IS NOT NULL
)
"""

BANK_TRANSACTIONS = f"""
SELECT
    NAMEORIG as "~from",
    NAMEDEST as "~to",
    TYPE AS "~label",
    STEP AS "step:Int",
    AMOUNT AS "amount:Float",
    OLDBALANCEORG AS "oldbalanceOrg:Float",
    NEWBALANCEORIG AS "newbalanceOrig:Float",
    OLDBALANCEDEST AS "oldbalanceDest:Float",
    NEWBALANCEDEST AS "newbalanceDest:Float",
    ISFRAUD AS "isFraud:Int",
    ISFLAGGEDFRAUD AS "isFlaggedFraud:Int"
FROM {snowflake_table} WHERE NAMEORIG IS NOT NULL AND NAMEDEST IS NOT NULL
"""

await session.import_from_table(
    graph,
    s3_location_import,
    [SOURCE_AND_DESTINATION_CUSTOMERS, BANK_TRANSACTIONS],
    catalog=athena_snowflake_catalog,
    database=snowflake_schema
)

## Execute Louvain Algorithm for Fraud Detection

Run Louvain community detection to identify potential fraud networks.

In [ ]:
# Verify graph data
all_nodes = graph.execute_query("MATCH (n) RETURN n LIMIT 10")
print(f"Sample nodes: {all_nodes}")

all_edges = graph.execute_query("MATCH ()-[r]-() RETURN r LIMIT 10")
print(f"Sample edges: {all_edges}")

In [ ]:
# Run Louvain algorithm and mutate graph with community property
louvain_result = graph.execute_query(
    'CALL neptune.algo.louvain.mutate({iterationTolerance:1e-07, writeProperty:"community"}) '
    'YIELD success AS success RETURN success'
)
print(f"Louvain result: {louvain_result}")

In [ ]:
# Find the top 10 communities by size
top_communities = graph.execute_query("""
MATCH (n)
WHERE n.community IS NOT NULL
RETURN n.community AS community, count(*) AS community_size
ORDER BY community_size DESC
LIMIT 10
""")
print(f"Top 10 communities: {top_communities}")

## Close session and delete resources

Ensure that you delete all resources, including:
* The S3 buckets used as a spill location
* The athena datasource catalog
* The athena lambda deployed as part of the data source
* The Neptune instance will be removed as part of the session (see `destroy_all_graphs()`)

In [ ]:
# Clean up
session.destroy_all_graphs()

## Conclusion

This notebook demonstrated:

1. **Snowflake Setup**: Uploaded PaySim data to Snowflake and created a table
2. **AWS Secrets Manager**: Stored Snowflake credentials securely
3. **Athena Data Source**: Created Snowflake data source in Athena (Lambda connector deployed automatically)
4. **Data Import**: Queried Snowflake via Athena and imported into Neptune Analytics
5. **Graph Analytics**: Ran Louvain community detection for fraud pattern identification

This architecture enables graph analytics on Snowflake data without ETL pipelines, combining Snowflake's data warehouse capabilities with Neptune Analytics' graph algorithms through Athena Federated Query.